# Airline Satisfaction Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify airline passenger **satisfaction** (satisfied vs neutral/dissatisfied) from survey ratings on in-flight services, demographics, and flight details. Synthetic dataset with ~10,000 rows and ~43% satisfied rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given passenger demographics, flight details, and service ratings (Wi-Fi, food, entertainment, seat comfort, etc.), predict whether the passenger is satisfied (1) or neutral/dissatisfied (0).

## 5 · Why This Project Matters

- Airlines compete heavily on **customer experience** — satisfaction drives loyalty.
- Identifying which service factors matter most enables targeted improvements.
- Survey-based classification is common in marketing analytics.
- This dataset teaches handling **ordinal survey ratings** as features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~10,000 |
| Features | 14 (age, flight_distance, class, type_of_travel, inflight_wifi, food_drink, seat_comfort, entertainment, onboard_service, leg_room, baggage_handling, cleanliness, departure_delay, arrival_delay) |
| Target | `satisfaction` (binary: 0=neutral/dissatisfied, 1=satisfied) |
| Class balance | ~57% dissatisfied, ~43% satisfied |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Airline Passenger Satisfaction survey (Kaggle).
**License**: Public / educational.
**Note**: Synthetic reproduction of passenger survey patterns.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "satisfaction"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
np.random.seed(SEED)
n = 10000

age = np.random.normal(39, 15, n).clip(7, 85).astype(int)
flight_distance = np.random.lognormal(6.5, 0.8, n).clip(50, 5000).astype(int)
travel_class = np.random.choice(['Business', 'Eco', 'Eco Plus'], n, p=[0.45, 0.4, 0.15])
type_of_travel = np.random.choice(['Business travel', 'Personal Travel'], n, p=[0.7, 0.3])
wifi = np.random.randint(1, 6, n)
food = np.random.randint(1, 6, n)
seat_comfort = np.random.randint(1, 6, n)
entertainment = np.random.randint(1, 6, n)
onboard_service = np.random.randint(1, 6, n)
leg_room = np.random.randint(1, 6, n)
baggage = np.random.randint(1, 6, n)
cleanliness = np.random.randint(1, 6, n)
dep_delay = np.random.exponential(15, n).clip(0, 300).astype(int)
arr_delay = (dep_delay + np.random.normal(0, 5, n)).clip(0, 320).astype(int)

avg_rating = (wifi + food + seat_comfort + entertainment + onboard_service + leg_room + baggage + cleanliness) / 8.0
score = (
    0.5 * (avg_rating - 3)
    + 0.4 * (travel_class == 'Business').astype(float)
    + 0.2 * (type_of_travel == 'Business travel').astype(float)
    - 0.01 * dep_delay
    + np.random.normal(0, 0.6, n)
)
satisfaction = (score > 0.3).astype(int)

df = pd.DataFrame({
    'age': age, 'flight_distance': flight_distance,
    'class': travel_class, 'type_of_travel': type_of_travel,
    'inflight_wifi': wifi, 'food_drink': food, 'seat_comfort': seat_comfort,
    'entertainment': entertainment, 'onboard_service': onboard_service,
    'leg_room': leg_room, 'baggage_handling': baggage, 'cleanliness': cleanliness,
    'departure_delay': dep_delay, 'arrival_delay': arr_delay, 'satisfaction': satisfaction,
})
print(f"Dataset shape: {df.shape}")
print(f"Satisfaction rate: {df['satisfaction'].mean():.2%}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
rating_cols = ['inflight_wifi', 'food_drink', 'seat_comfort', 'entertainment',
               'onboard_service', 'leg_room', 'baggage_handling', 'cleanliness']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(rating_cols):
    ax = axes[i // 4, i % 4]
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'coral'], legend=False)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('Rating')
plt.suptitle("Satisfaction Rate by Service Rating", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['class', 'type_of_travel']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Satisfaction by {col}")
plt.tight_layout()
plt.show()
print(f"Rating columns: {len(rating_cols)}")

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Satisfaction Distribution")
axes[0].set_xticklabels(['Dissatisfied (0)', 'Satisfied (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()
rating_cols = ['inflight_wifi', 'food_drink', 'seat_comfort', 'entertainment',
               'onboard_service', 'leg_room', 'baggage_handling', 'cleanliness']
X_train['avg_rating'] = X_train[rating_cols].mean(axis=1)
X_test['avg_rating'] = X_test[rating_cols].mean(axis=1)
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Logistic Regression as baseline.

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [ ]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

## 25 · Interpretation and Business Insight

- **Travel class (Business)** is the strongest predictor of satisfaction.
- **Service ratings** (wifi, entertainment, seat comfort) collectively drive satisfaction.
- **Departure delay** reduces satisfaction but less than service quality.
- **Business travelers** are more likely to be satisfied than personal travelers.
- Average rating across all services is a powerful engineered feature.

## 26 · Limitations

1. Synthetic/survey data — self-reported satisfaction is subjective.
2. Rating scales (1-5) may not capture nuance.
3. No airline identity or route information.
4. Departure and arrival delays are highly correlated (redundant).
5. Missing important factors: price paid, frequent flyer status.

## 27 · How to Improve This Project

1. Add ticket price and loyalty tier features.
2. Segment analysis by route length (short vs long haul).
3. Use ordinal encoding that respects rating order.
4. Build a recommendation system (which service to improve first).
5. Add NLP from free-text reviews for richer signals.

## 28 · Production Considerations

- Post-flight satisfaction prediction for proactive service recovery.
- Integration with CRM for targeted loyalty offers.
- Real-time dashboards for airline service quality monitoring.
- A/B testing framework for service improvements.

## 29 · Common Mistakes

1. Treating ordinal ratings as purely categorical.
2. Not aggregating ratings into summary statistics.
3. Including both departure and arrival delay (high collinearity).
4. Ignoring the strong class × satisfaction interaction.
5. Not segmenting by travel type for targeted insights.

## 30 · Mini Challenge / Exercises

1. Remove all rating columns and use only flight/demographic features — how much accuracy drops?
2. Build separate models for Business vs Eco class.
3. Find which single service rating matters most for satisfaction.
4. Compare treating ratings as numeric vs categorical.

## 31 · Final Summary / Key Takeaways

1. Airline satisfaction is driven by service quality and travel class.
2. Business class passengers are significantly more satisfied.
3. Average service rating is a powerful summary feature.
4. Delays matter less than in-flight service quality.
5. Targeted service improvements need segment-specific analysis.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))